In [23]:
pip install timm torchvision torch

Note: you may need to restart the kernel to use updated packages.


In [24]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

In [3]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageDraw
from torchvision import transforms

class WarpBBoxDataset(Dataset):
    def __init__(self, image_dir, label_dir, num_classes=28, img_size=224):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_paths = [os.path.join(image_dir, img) for img in os.listdir(image_dir)]
        self.num_classes = num_classes
        self.img_size = img_size
        self.rgb_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        filename = os.path.splitext(os.path.basename(image_path))[0]
        label_path = os.path.join(self.label_dir, filename + '.txt')

        # Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        rgb_tensor = self.rgb_transform(image)

        # Initialize mask and label vector
        mask = Image.new('L', (self.img_size, self.img_size), 0)
        label = torch.zeros(self.num_classes)

        # Add bounding boxes to the mask
        if os.path.exists(label_path):
            w_orig, h_orig = image.size
            with open(label_path, 'r') as f:
                draw = ImageDraw.Draw(mask)
                for line in f:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    label[int(class_id)] = 1

                    # Convert YOLO format to box corners
                    x1 = (x_center - width / 2) * self.img_size
                    y1 = (y_center - height / 2) * self.img_size
                    x2 = (x_center + width / 2) * self.img_size
                    y2 = (y_center + height / 2) * self.img_size

                    draw.rectangle([x1, y1, x2, y2], fill=255)

        # Convert mask to tensor and normalize
        mask_tensor = transforms.ToTensor()(mask)

        # Concatenate image and mask: (3 + 1) channels
        combined_input = torch.cat([rgb_tensor, mask_tensor], dim=0)

        return combined_input, label


In [1]:
import timm
import torch.nn as nn

model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True)

# Modify patch embedding layer for 4-channel input (RGB + mask)
patch_embed = model.patch_embed
patch_embed.proj = nn.Conv2d(4, patch_embed.proj.out_channels,
                             kernel_size=patch_embed.proj.kernel_size,
                             stride=patch_embed.proj.stride,
                             padding=patch_embed.proj.padding)

import torch.nn.functional as F

class MultiLabelSwinHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x shape: [batch_size, num_patches, embed_dim]
        x = x.mean(dim=1)  # Global average pooling over patch tokens
        x = self.classifier(x)
        return torch.sigmoid(x)

model.head = MultiLabelSwinHead(model.head.in_features, 28)


C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [5]:
from torch.utils.data import DataLoader
train_dataset = WarpBBoxDataset(
    image_dir=r"C:\Users\Alfina\Downloads\archive\Warp-D\train\images",
    label_dir=r"C:\Users\Alfina\Downloads\archive\Warp-D\train\labels",
    num_classes=28
)

val_dataset = WarpBBoxDataset(
    image_dir=r"C:\Users\Alfina\Downloads\archive\Warp-D\test\images",
    label_dir=r"C:\Users\Alfina\Downloads\archive\Warp-D\test\labels",
    num_classes=28
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [6]:
class SwinMultiLabelWrapper(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.backbone = base_model
        self.classifier = nn.Linear(base_model.num_features, num_classes)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return self.activation(x)


In [7]:
import torch.nn as nn
from timm import create_model

class MultiLabelSwin(nn.Module):
    def __init__(self, num_classes=28):
        super().__init__()
        self.backbone = create_model("swin_tiny_patch4_window7_224", pretrained=True, features_only=True)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # Global Average Pooling
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.backbone.feature_info[-1]['num_chs'], num_classes),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Feature map from the last stage
        x = self.backbone(x)[-1]  # shape: [B, C, 7, 7]
        x = self.pool(x)          # shape: [B, C, 1, 1]
        x = self.classifier(x)    # shape: [B, 28]
        return x

model = MultiLabelSwin(num_classes=28)

In [8]:
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [9]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import numpy as np
from sklearn.metrics import f1_score

class WarpBBoxDataset(Dataset):
    def __init__(self, image_dir, label_dir, num_classes=28, img_size=224):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_paths = [os.path.join(image_dir, img) for img in os.listdir(image_dir) if img.endswith(".jpg")]
        self.num_classes = num_classes
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = os.path.join(self.label_dir, os.path.splitext(os.path.basename(image_path))[0] + ".txt")

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        label = torch.zeros(self.num_classes)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    if line.strip() == "":
                        continue
                    try:
                        class_id, x_center, y_center, width, height = map(float, line.strip().split())
                        label[int(class_id)] = 1
                    except ValueError:
                        print(f"⚠️ Skipping malformed line in {label_path}: '{line.strip()}'")

        return image, label

def calculate_f1_score(model, val_loader, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = (outputs > 0.5).float()
            all_preds.append(preds.cpu())
            all_targets.append(labels.cpu())

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    f1 = f1_score(all_targets.numpy(), all_preds.numpy(), average='macro')
    print(f"📊 Validation F1 Score: {f1:.4f}")
    return f1


In [25]:
from timm import create_model
import torch.nn as nn
import torch

class MultiLabelSwin(nn.Module):
    def __init__(self, num_classes=28):
        super().__init__()
        # Get Swin without classification head
        self.backbone = create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Linear(768, num_classes),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.backbone(x)       # [B, 768]
        x = self.classifier(x)     # [B, 28]
        return x

In [11]:
model = MultiLabelSwin(num_classes=28).to(device)

In [31]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class WasteDataset(Dataset):
    def __init__(self, image_paths, label_paths, transform=None, num_classes=28):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.transform = transform
        self.num_classes = num_classes

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Initialize multi-label vector
        labels = torch.zeros(self.num_classes)

        # Safely read label file
        label_file = self.label_paths[idx]
        if os.path.exists(label_file):
            with open(label_file, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue  # skip blank lines

                    parts = line.split()
                    if len(parts) != 5:
                        print(f"⚠️ Skipping malformed line in {label_file}: {line}")
                        continue

                    try:
                        class_id = int(float(parts[0]))
                        if 0 <= class_id < self.num_classes:
                            labels[class_id] = 1
                    except ValueError:
                        print(f"❌ Invalid class ID in {label_file}: {parts[0]}")
                        continue

        return image, labels

In [27]:
import os

label_dir = r"C:\Users\Alfina\Downloads\archive\Warp-D\train\labels"

for fname in os.listdir(label_dir):
    path = os.path.join(label_dir, fname)
    with open(path, 'r') as f:
        lines = f.readlines()

    valid_lines = [line for line in lines if len(line.strip().split()) == 5]

    with open(path, 'w') as f:
        f.writelines(valid_lines)


In [28]:
from PIL import Image

class WasteDataset(Dataset):
    def __init__(self, image_paths, label_paths, transform=None, num_classes=28):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.transform = transform
        self.num_classes = num_classes

    def __getitem__(self, idx):
        # 🔐 Force RGB conversion to avoid 4-channel error
        img = Image.open(self.image_paths[idx])
        if img.mode != 'RGB':
            img = img.convert('RGB')

        if self.transform:
            img = self.transform(img)

        labels = torch.zeros(self.num_classes)
        label_path = self.label_paths[idx]

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        try:
                            class_id = int(float(parts[0]))
                            if 0 <= class_id < self.num_classes:
                                labels[class_id] = 1
                        except:
                            continue

        return img, labels

    def __len__(self):
        return len(self.image_paths)

In [32]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [30]:
dataset = WasteDataset(image_paths, label_paths, transform=your_transform)

# Test one sample
img, label = dataset[0]

print("Image shape:", img.shape)  # should be [3, H, W]
print("Label vector:", label)

NameError: name 'image_paths' is not defined

In [20]:
best_f1 = 0.0
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)  # [B, 28]
        loss = criterion(outputs, labels.float())  # For BCEWithLogitsLoss or BCELoss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    print(f"\n📦 Epoch [{epoch + 1}/{num_epochs}] - 🏋️ Train Loss: {avg_train_loss:.4f}")

    # 🔍 Evaluate F1 Score on Validation Set
    model.eval()
    with torch.no_grad():
        f1 = calculate_f1_score(model, val_loader, device)

    print(f"📈 Validation F1 Score: {f1:.4f}")

    # 💾 Save the best model
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), r"C:\wasteproject\best_swin_model.pth")
        print("✅ New best model saved with F1:", f1)


RuntimeError: Given groups=1, weight of size [96, 3, 4, 4], expected input[32, 4, 224, 224] to have 3 channels, but got 4 channels instead